# Lab 06 - Few-Shot Prompting

## Curating Examples, Ordering Strategies, and Counter-Examples

**Week 3 - Prompt Engineering and Task-to-Prompt Mapping**

You already know that a clear instruction and a strict output contract make a prompt reliable
(Labs 01 to 03). This lab adds the next lever: **in-context examples**. You will build the
few-shot machinery an engineer actually owns, which is curation, ordering, template assembly,
output parsing, and offline scoring, then measure whether few-shot beats zero-shot on a fixed
evaluation set.

The classification target is **app-review intent** with five labels:
`bug_report`, `feature_request`, `praise`, `question`, `other`.

### Outcomes

By the end of this lab you will be able to:

1. State what makes few-shot distinct from zero-shot and when it pays off.
2. Curate examples across three roles (typical, edge, adversarial) with a reproducible rule.
3. Build three orderings (easy to hard, similar first, interleaved) and compare them.
4. Assemble a few-shot template with delimiters and a strict JSON output contract.
5. Parse messy model output and score it against real gold labels (precision, recall, F1).
6. Reason about trade-offs: example count versus context budget, coverage gaps, and stability.

### How this lab runs

Every cell is offline, deterministic, and standard-library only. The model is a provided
stand-in named `simulated_llm`. It behaves like a hosted classifier: it reads the examples in
your prompt, returns raw text with a JSON block, and gives the **same answer every run** so the
numbers are reproducible in class. The graded skill is the engineering around the model, not the
model itself. An appendix shows how to swap in a real provider call.

> Work top to bottom. Each `TODO` raises `NotImplementedError` until you implement it. The
> `check(...)` cells are soft: they print PASS, FAIL, or ERROR and never stop the notebook.

## Part 0 - Setup (provided)

Run these cells as is. They define the data, the simulated model, and the `check` helper.
Do not edit them.

In [ ]:
import re
import json
from collections import Counter, defaultdict

LABELS = ["bug_report", "feature_request", "praise", "question", "other"]

### Data (synthetic, clearly fictional)

`GOLD` holds a small labeled training pool and a held-out evaluation set. Unlike a first-draft
lab, the eval items carry **true gold labels**, so scoring is honest rather than circular.

`POOL` is a bank of 14 candidate examples tagged by role: `typical` (clear signal),
`edge` (subtle distinction), and `adversarial` (a counter-example that carves a boundary).

In [ ]:
GOLD = {
    "labels": LABELS,
    "train": [
        {"id": "T01", "text": "Crashes when I open settings on Android 14", "label": "bug_report"},
        {"id": "T02", "text": "Please add a dark mode schedule option", "label": "feature_request"},
        {"id": "T03", "text": "Love the new UI, so clean and fast!", "label": "praise"},
        {"id": "T04", "text": "How do I export my data to CSV?", "label": "question"},
        {"id": "T05", "text": "Keyboard stops responding after last update", "label": "bug_report"},
    ],
    "eval": [
        {"id": "E01", "text": "Since 2.1 update, it freezes on startup", "label": "bug_report"},
        {"id": "E02", "text": "Could you support widgets on the home screen?", "label": "feature_request"},
        {"id": "E03", "text": "Thanks for fixing sync, works perfectly now!", "label": "praise"},
        {"id": "E04", "text": "Is there any way to change the font size?", "label": "question"},
        {"id": "E05", "text": "Sometimes notifications don't arrive until hours later", "label": "bug_report"},
        {"id": "E06", "text": "Good app but the ads make it almost unusable", "label": "other"},
        {"id": "E07", "text": "Great app, devs! Keep it up!", "label": "praise"},
        {"id": "E08", "text": "Where can I find the changelog?", "label": "question"},
        {"id": "E09", "text": "Please let me color-code playlists", "label": "feature_request"},
        {"id": "E10", "text": "It worked yesterday; today it won't open the camera", "label": "bug_report"},
    ],
}

POOL = {
    "candidates": [
        {"id": "S01", "text": "The app crashes when I tap Share", "label": "bug_report", "category": "typical"},
        {"id": "S02", "text": "Could you add export to PDF?", "label": "feature_request", "category": "typical"},
        {"id": "S03", "text": "Love this app, best calendar ever", "label": "praise", "category": "typical"},
        {"id": "S04", "text": "How can I disable sounds?", "label": "question", "category": "typical"},
        {"id": "S05", "text": "Sync is slow sometimes but it recovers on its own", "label": "other", "category": "edge"},
        {"id": "S06", "text": "On iPad mini the screen flickers when rotating", "label": "bug_report", "category": "edge"},
        {"id": "S07", "text": "Please support keyboard shortcuts on desktop", "label": "feature_request", "category": "edge"},
        {"id": "S08", "text": "Notifications sometimes arrive an hour late", "label": "bug_report", "category": "edge"},
        {"id": "S09", "text": "Where are templates stored?", "label": "question", "category": "edge"},
        {"id": "S10", "text": "Thanks for fixing the login bug, working great now", "label": "praise", "category": "adversarial"},
        {"id": "S11", "text": "Add themes? Actually never mind, ignore me", "label": "other", "category": "adversarial"},
        {"id": "S12", "text": "Nice app overall but the constant ads ruin it", "label": "other", "category": "adversarial"},
        {"id": "S13", "text": "Why is there still no Linux build?", "label": "question", "category": "adversarial"},
        {"id": "S14", "text": "It crashes unless I log out first", "label": "bug_report", "category": "adversarial"},
    ]
}

EVAL = GOLD["eval"]
GOLD_MAP = {e["id"]: e["label"] for e in EVAL}
print(f"eval items: {len(EVAL)} | pool candidates: {len(POOL['candidates'])} | labels: {LABELS}")

### The simulated model (provided)

`simulated_llm(examples, eval_items)` is a deterministic stand-in for a hosted LLM. It reads the
examples you place in the prompt, scores each eval item, and returns **raw text** with a fenced
JSON block, exactly the kind of output a real model emits. It is sensitive to which examples you
include and to their order, so your curation and ordering choices show up in the results.

You will not edit this cell. Treat it as an opaque endpoint you send prompts to.

In [ ]:
_LEXICON = {
    "bug_report":      ["crash", "crashes", "freeze", "freezes", "flicker", "flickers", "broken",
                        "error", "won't", "wont", "stops", "stopped", "fails", "bug", "fix", "fixing", "fixed"],
    "feature_request": ["add", "support", "could you add", "please add", "import", "export",
                        "shortcut", "shortcuts", "widget", "widgets", "option", "ability", "let me"],
    "praise":          ["love", "great", "amazing", "best", "awesome", "thanks", "perfect",
                        "perfectly", "clean", "fast", "good"],
    "question":        ["how", "where", "when", "why", "which", "is there", "can i", "?"],
    "other":           [],
}
_PRIORITY = ["bug_report", "feature_request", "question", "praise", "other"]
_ALPHA = 3.0    # how strongly a similar example pulls the decision
_BETA = 0.15    # mild recency: later examples in the prompt weigh a little more

def _tok(s):
    return set(re.findall(r"[a-z0-9']+", (s or "").lower()))

def _jacc(a, b):
    u = a | b
    return (len(a & b) / len(u)) if u else 0.0

def _base_scores(text):
    low = (text or "").lower()
    sc = {L: 0.0 for L in LABELS}
    for L, kws in _LEXICON.items():
        for kw in kws:
            if kw in low:
                sc[L] += 1.0
    return sc

def _decide(sc):
    if max(sc.values()) <= 0.0:
        return "other", 0.0
    ranked = sorted(LABELS, key=lambda L: (-sc[L], _PRIORITY.index(L)))
    runner = sc[ranked[1]] if len(ranked) > 1 else 0.0
    return ranked[0], sc[ranked[0]] - runner

def simulated_llm(examples, eval_items):
    """Deterministic stand-in for a hosted LLM classification call.
    examples: list of {'label','text',...} in PROMPT ORDER (empty list == zero-shot).
    Returns RAW TEXT: a short preamble followed by a fenced JSON block."""
    n = len(examples)
    ex_tok = [(_tok(e["text"]), e["label"], e.get("id", "?")) for e in examples]
    records = []
    for item in eval_items:
        sc = _base_scores(item["text"])
        itoks = _tok(item["text"])
        nearest, best_sim = None, 0.0
        for idx, (etoks, elabel, eid) in enumerate(ex_tok):
            w = 1.0 + _BETA * (idx / (n - 1)) if n > 1 else 1.0
            sim = _jacc(itoks, etoks)
            sc[elabel] += _ALPHA * w * sim
            if sim > best_sim:
                best_sim, nearest = sim, eid
        label, margin = _decide(sc)
        if examples and nearest is not None and best_sim > 0:
            reason = f"nearest example {nearest} (sim={best_sim:.2f}); margin={margin:.2f}"
        else:
            reason = f"zero-shot lexical match; margin={margin:.2f}"
        records.append({"id": item["id"], "label": label, "reasons": [reason]})
    doc = {"records": records, "stats": {"count": len(records)}}
    return "Here are the classifications:\n```json\n" + json.dumps(doc, ensure_ascii=False) + "\n```\n"

print("simulated_llm ready. Raw output preview:")
print(simulated_llm([], EVAL[:1]))

### The `check(...)` helper (provided)

`check` is a soft assertion. It prints PASS, FAIL, or ERROR, tallies the totals, and never
raises, so an unimplemented `TODO` shows up as ERROR instead of crashing the notebook. Aim to
turn every check green.

In [ ]:
_TALLY = {"pass": 0, "fail": 0, "error": 0}

def check(label, cond, detail=""):
    """Soft assertion. `cond` may be a bool or a zero-arg callable."""
    try:
        ok = cond() if callable(cond) else bool(cond)
    except NotImplementedError:
        _TALLY["error"] += 1
        print(f"\u26a0\ufe0f  ERROR  {label}  (not implemented yet)")
        return False
    except Exception as e:
        _TALLY["error"] += 1
        print(f"\u26a0\ufe0f  ERROR  {label}\n        -> {type(e).__name__}: {e}")
        return False
    if ok:
        _TALLY["pass"] += 1
        print(f"\u2705 PASS  {label}")
    else:
        _TALLY["fail"] += 1
        line = f"\u274c FAIL  {label}"
        if detail:
            line += f"\n        -> {detail}"
        print(line)
    return bool(ok)

def summary():
    t = _TALLY
    print(f"\n==== {t['pass']} PASS | {t['fail']} FAIL | {t['error']} ERROR ====")

check("check() helper is live", True)

## Part A - Warm-up: tokens and similarity

Everything downstream (curation, similar-first ordering) rests on a token set and a Jaccard
overlap. Build both. Keep them pure and side-effect free.

In [ ]:
def tokenize(text):
    """Return the set of lowercase word tokens in `text`.
    A token is a maximal run of [a-z0-9'] after lowercasing. None -> empty set."""
    return set(re.findall(r"[a-z0-9']+", (text or "").lower()))

In [ ]:
def jaccard(a, b):
    """Jaccard similarity of two sets: |a & b| / |a | b|. Empty over empty is 0.0."""
    u = a | b
    return (len(a & b) / len(u)) if u else 0.0

In [ ]:
def corpus_tokens(items):
    """Union of tokens across a list of {'text': ...} items. Provided; uses your tokenize()."""
    out = set()
    for it in items:
        out |= tokenize(it["text"])
    return out

In [ ]:
check("tokenize splits and lowercases",
      lambda: tokenize("Hello, WORLD! it's 2.1") == {"hello", "world", "it's", "2", "1"},
      "expected {'hello','world',\"it's\",'2','1'}")
check("tokenize handles None", lambda: tokenize(None) == set())
check("jaccard basic", lambda: abs(jaccard({1, 2, 3}, {2, 3, 4}) - 0.5) < 1e-9, "expected 0.5")
check("jaccard identical", lambda: jaccard({"a", "b"}, {"a", "b"}) == 1.0)
check("jaccard empty over empty", lambda: jaccard(set(), set()) == 0.0)

## Part B - Zero-shot baseline

Before adding examples, measure the model with instruction only. You need three pieces:

1. `build_prompt` to assemble the prompt text (it must also work with an empty example list).
2. `parse_output` to pull the JSON object out of the model's raw text.
3. `score` to compute per-label precision, recall, F1, plus a macro row, against gold.

Then run the zero-shot variant and record its macro F1. That number is the bar few-shot must clear.

In [ ]:
def build_prompt(examples, eval_items):
    """Assemble a classification prompt as a single string.

    Required blocks, in order:
      <INSTRUCTION> ... </INSTRUCTION>
      <OUTPUT_CONTRACT> ... </OUTPUT_CONTRACT>
      <EXAMPLES> ... </EXAMPLES>   (only when `examples` is non-empty)
      <QUERY> ... </QUERY>

    Rules:
      - The instruction lists the five labels from LABELS.
      - The contract shows the strict JSON shape the model must return.
      - Each example renders on its own line as:  [label] :: text
      - Each eval item renders on its own line as: id :: text
    """
    lines = ["<INSTRUCTION>",
             "Classify each review into one of: " + ", ".join(LABELS) + "."]
    if examples:
        lines.append("Use the EXAMPLES before the QUERY to learn the label boundaries.")
    lines += ["Return STRICT JSON only, following OUTPUT_CONTRACT. No extra prose.",
              "</INSTRUCTION>\n",
              "<OUTPUT_CONTRACT>",
              '{"records":[{"id":"<E##>","label":"<one_of_labels>","reasons":["<why>"]}],"stats":{"count":<int>}}',
              "</OUTPUT_CONTRACT>\n"]
    if examples:
        lines.append("<EXAMPLES>")
        for e in examples:
            lines.append(f"[{e['label']}] :: {e['text']}")
        lines.append("</EXAMPLES>\n")
    lines.append("<QUERY>")
    for it in eval_items:
        lines.append(f"{it['id']} :: {it['text']}")
    lines.append("</QUERY>")
    return "\n".join(lines)

In [ ]:
def parse_output(raw):
    """Return the JSON object embedded in the model's raw text.
    The model may add a prose preamble and wrap the JSON in a fenced block."""
    text = raw.strip()
    if "```" in text:
        block = text.split("```")[1]
        if block.lstrip().lower().startswith("json"):
            block = block.split("\n", 1)[1] if "\n" in block else block
        text = block.strip()
    return json.loads(text)

In [ ]:
def score(gold, pred):
    """Per-label precision, recall, F1, plus a macro row.
    gold, pred: dict mapping item id -> label. A missing prediction counts as 'other'.
    Returns: dict {label: {'precision','recall','f1'}, 'MACRO': {...}}.
    Macro is the unweighted mean over every label seen in gold or pred."""
    labels = sorted(set(gold.values()) | set(pred.values()))
    cm = defaultdict(Counter)
    for _id, t in gold.items():
        cm[t][pred.get(_id, "other")] += 1
    rows, macro = {}, []
    for L in labels:
        tp = cm[L][L]
        fp = sum(cm[x][L] for x in labels if x != L)
        fn = sum(cm[L][x] for x in labels if x != L)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = (2 * p * r / (p + r)) if p + r else 0.0
        rows[L] = {"precision": p, "recall": r, "f1": f}
        macro.append((p, r, f))
    m = len(macro)
    rows["MACRO"] = {"precision": sum(x[0] for x in macro) / m,
                     "recall": sum(x[1] for x in macro) / m,
                     "f1": sum(x[2] for x in macro) / m}
    return rows

### Run the zero-shot variant

`run_variant` (provided) wires your four functions into one pipeline: build the prompt, call the
model, parse, validate, and return predictions. `validate_output` arrives in Part E, so a light
inline guard is used here.

In [ ]:
def run_variant(examples, eval_items):
    """Provided driver. Uses your build_prompt / parse_output; returns (doc, pred, prompt)."""
    prompt = build_prompt(examples, eval_items)
    doc = parse_output(simulated_llm(examples, eval_items))
    pred = {r["id"]: r["label"] for r in doc["records"]}
    return doc, pred, prompt

def show_scores(title, s):
    print(title)
    for L in LABELS + ["MACRO"]:
        row = s[L]
        print(f"  {L:16} P={row['precision']:.3f} R={row['recall']:.3f} F1={row['f1']:.3f}")

In [ ]:
def _b_prompt_zero():
    p = build_prompt([], EVAL)
    return ("<QUERY>" in p and "<OUTPUT_CONTRACT>" in p and "<EXAMPLES>" not in p
            and all(e["id"] in p for e in EVAL))
def _b_prompt_few():
    p = build_prompt([POOL["candidates"][0]], EVAL)
    return "<EXAMPLES>" in p and "[bug_report] :: The app crashes when I tap Share" in p
check("build_prompt zero-shot has QUERY + contract, no EXAMPLES", _b_prompt_zero)
check("build_prompt few-shot renders [label] :: text", _b_prompt_few)

def _b_parse():
    doc = parse_output(simulated_llm([], EVAL))
    return isinstance(doc, dict) and len(doc["records"]) == 10 and doc["stats"]["count"] == 10
check("parse_output returns 10 records", _b_parse)

def _b_score():
    _, pred0, _ = run_variant([], EVAL)
    s = score(GOLD_MAP, pred0)
    return round(s["MACRO"]["f1"], 3) == 0.720
check("zero-shot macro F1 == 0.720", _b_score, "check score() and build/parse")

In [ ]:
try:
    doc0, pred0, _ = run_variant([], EVAL)
    s0 = score(GOLD_MAP, pred0)
    show_scores("ZERO-SHOT scores:", s0)
    wrong0 = [r["id"] for r in doc0["records"] if r["label"] != GOLD_MAP[r["id"]]]
    print("\nZero-shot mistakes:", wrong0)
    for rid in wrong0:
        e = next(x for x in EVAL if x["id"] == rid)
        print(f"  {rid} predicted {pred0[rid]:16} gold {GOLD_MAP[rid]:16} :: {e['text']}")
except Exception as ex:
    print(f"[implement the Part A and Part B TODOs, then re-run] {type(ex).__name__}: {ex}")

## Part C - Curate few-shot examples

You have 14 candidates across three roles. Pick **2 typical, 2 edge, 2 adversarial** with a rule
you can reproduce and defend, rather than by taste. The rule here: within each role, rank
candidates by how representative they are of the eval vocabulary, then take the top two.

In [ ]:
def curate(pool, eval_items, per_category=2):
    """Select `per_category` examples from each role: typical, edge, adversarial.

    Within a role, rank candidates by Jaccard similarity between the candidate's tokens and the
    union of eval tokens (corpus_tokens(eval_items)), descending. Break ties by id ascending.
    Return the picks concatenated in role order: typical, then edge, then adversarial.
    """
    ct = corpus_tokens(eval_items)
    by_cat = defaultdict(list)
    for c in pool["candidates"]:
        by_cat[c["category"]].append(c)
    picked = []
    for cat in ("typical", "edge", "adversarial"):
        ranked = sorted(by_cat[cat], key=lambda c: (-jaccard(tokenize(c["text"]), ct), c["id"]))
        picked.extend(ranked[:per_category])
    return picked

In [ ]:
def _c_shape():
    cur = curate(POOL, EVAL)
    cats = Counter(c["category"] for c in cur)
    return len(cur) == 6 and cats["typical"] == 2 and cats["edge"] == 2 and cats["adversarial"] == 2
def _c_ids():
    cur = curate(POOL, EVAL)
    return {c["id"] for c in cur} == {"S01", "S02", "S05", "S07", "S10", "S12"}
check("curate returns 2 per role (6 total)", _c_shape)
check("curate selects the expected ids", _c_ids, "expected S01,S02,S05,S07,S10,S12")

In [ ]:
try:
    CURATED = curate(POOL, EVAL)
    print("Curated few-shot set:")
    for c in CURATED:
        print(f"  {c['id']} [{c['category']:11}] {c['label']:16} {c['text']}")
except Exception as ex:
    CURATED = None
    print(f"[implement curate(), then re-run] {type(ex).__name__}: {ex}")

**Reflection (write two or three sentences).** For each of the two adversarial picks, name the
boundary it is meant to carve. Which eval item do you expect it to fix, and why?

## Part D - Ordering strategies and counter-examples

Same six examples, three orders. Build each ordering, then compare. Ordering is a real lever:
on clear inputs the examples dominate and order barely moves the label, but on borderline inputs
the order can decide the outcome. You will see both effects.

In [ ]:
_DIFF = {"typical": 0, "edge": 1, "adversarial": 2}

def order_easy_to_hard(examples):
    """Sort by role difficulty (typical < edge < adversarial), ties by id ascending."""
    return sorted(examples, key=lambda c: (_DIFF[c["category"]], c["id"]))

In [ ]:
def order_similar_first(examples, eval_items):
    """Sort by Jaccard similarity to the eval corpus, descending; ties by id ascending."""
    ct = corpus_tokens(eval_items)
    return sorted(examples, key=lambda c: (-jaccard(tokenize(c["text"]), ct), c["id"]))

In [ ]:
def order_interleave(examples):
    """Round-robin across roles in the cycle typical, edge, adversarial.
    Within each role, keep id order. Drain one role per turn until all are placed."""
    buckets = defaultdict(list)
    for c in examples:
        buckets[c["category"]].append(c)
    for cat in buckets:
        buckets[cat].sort(key=lambda c: c["id"])
    out, cats, i = [], ["typical", "edge", "adversarial"], 0
    while len(out) < len(examples):
        cat = cats[i % 3]
        if buckets[cat]:
            out.append(buckets[cat].pop(0))
        i += 1
    return out

In [ ]:
check("easy_to_hard order",
      lambda: [c["id"] for c in order_easy_to_hard(curate(POOL, EVAL))] == ["S01", "S02", "S05", "S07", "S10", "S12"])
check("similar_first order",
      lambda: [c["id"] for c in order_similar_first(curate(POOL, EVAL), EVAL)] == ["S10", "S05", "S12", "S02", "S07", "S01"])
check("interleave order (T,E,A,...)",
      lambda: [c["id"] for c in order_interleave(curate(POOL, EVAL))] == ["S01", "S05", "S10", "S02", "S07", "S12"])

### Ordering-sensitivity probe (provided)

The three orderings above agree on labels for the clean eval set, so the effect of order is easy
to miss. This probe isolates it. The review below is a genuine tie: `love` votes praise and
`crash` votes bug, and the two demo examples are equally similar to it. With everything else
equal, the model leans toward the example it saw **last** (a mild recency effect). Flip the order,
flip the label.

In [ ]:
def _probe_parse(raw):
    return json.loads(raw.split("```json", 1)[1].split("```", 1)[0])

demo_praise = {"id": "DA", "label": "praise",     "text": "I love it"}
demo_bug    = {"id": "DB", "label": "bug_report", "text": "it will crash"}
probe = [{"id": "P01", "text": "I love it but it will crash"}]

for order, name in [([demo_bug, demo_praise], "praise example LAST"),
                    ([demo_praise, demo_bug], "bug example LAST")]:
    rec = _probe_parse(simulated_llm(order, probe))["records"][0]
    print(f"  {name:22} -> {rec['label']:12} ({rec['reasons'][0]})")
print("\nTakeaway: with equal evidence, the last example wins. Order matters on borderline inputs.")

## Part E - Evaluate, validate, and compare

Add one more control, `validate_output`, so a malformed model response fails loudly instead of
poisoning the score. Then run all three orderings and build a comparison across macro F1 and mean
decision margin (a rough confidence proxy pulled from the model's `reasons`).

In [ ]:
def validate_output(doc, n_expected=10):
    """Assert the model response is well formed. Raise ValueError on any violation, else True.
    Checks: dict with 'records' and 'stats'; exactly n_expected records; stats.count == n_expected;
    every label in LABELS; every record has a non-empty 'reasons' list."""
    if not (isinstance(doc, dict) and "records" in doc and "stats" in doc):
        raise ValueError("missing top-level records/stats")
    recs = doc["records"]
    if len(recs) != n_expected or doc["stats"].get("count") != n_expected:
        raise ValueError(f"expected {n_expected} records")
    allowed = set(LABELS)
    for r in recs:
        if r.get("label") not in allowed:
            raise ValueError(f"bad label {r.get('label')!r}")
        if not (isinstance(r.get("reasons"), list) and r["reasons"]):
            raise ValueError("reasons must be a non-empty list")
    return True

In [ ]:
check("validate_output accepts a good doc",
      lambda: validate_output(parse_output(simulated_llm([], EVAL))) is True)

def _raises(bad):
    try:
        validate_output(bad); return False
    except ValueError:
        return True
check("validate_output rejects wrong count", lambda: _raises({"records": [], "stats": {"count": 0}}))
check("validate_output rejects bad label",
      lambda: _raises({"records": [{"id": "E01", "label": "nope", "reasons": ["x"]}] * 10,
                       "stats": {"count": 10}}))

In [ ]:
def mean_margin(doc):
    ms = [float(r["reasons"][0].split("margin=")[1]) for r in doc["records"]]
    return sum(ms) / len(ms)

rows = {}
try:
    orders = {
        "zero_shot":     [],
        "easy_to_hard":  order_easy_to_hard(CURATED),
        "similar_first": order_similar_first(CURATED, EVAL),
        "interleave":    order_interleave(CURATED),
    }
    print(f"{'variant':14} {'macro_F1':>9} {'mean_margin':>12}   wrong")
    for name, ex in orders.items():
        doc, pred, _ = run_variant(ex, EVAL)
        validate_output(doc)
        s = score(GOLD_MAP, pred)
        wrong = [r["id"] for r in doc["records"] if r["label"] != GOLD_MAP[r["id"]]]
        rows[name] = (s["MACRO"]["f1"], mean_margin(doc), wrong)
        print(f"{name:14} {s['MACRO']['f1']:9.3f} {mean_margin(doc):12.3f}   {wrong}")
except Exception as ex:
    print(f"[implement the Part C, D, and E TODOs, then re-run] {type(ex).__name__}: {ex}")

In [ ]:
check("few-shot beats zero-shot", lambda: rows["easy_to_hard"][0] > rows["zero_shot"][0])
check("few-shot macro F1 == 0.893", lambda: round(rows["easy_to_hard"][0], 3) == 0.893)
check("residual error is exactly E05", lambda: rows["easy_to_hard"][2] == ["E05"])

### Read the residual

Few-shot lifts macro F1 from 0.720 to 0.893 and fixes E06 (the mixed-sentiment review) using the
adversarial `other` counter-example. One error survives: **E05**, the silent notification delay.
The disambiguating example exists in the pool (S08, a delayed-notification bug), but the
similarity-based curation ranked it just below the cut, so the model never saw it. That is the
lesson: generic curation optimizes for vocabulary overlap, not for coverage of your hardest
boundaries. Stretch 1 fixes it directly.

In [ ]:
summary()

## Stretch goals

These are optional and harder. In the student notebook they are stubs; the solution notebook
implements all three.

### Stretch 1 - Targeted counter-example injection

Curation missed the example that fixes E05. Write `curate_with_coverage` that starts from the
similarity-curated six and, if any eval error class is uncovered, swaps in a targeted example.
Here: append the delayed-notification bug (S08) and confirm macro F1 reaches 1.000.

In [ ]:
def curate_with_coverage(pool, eval_items):
    """Similarity curation, then guarantee the delayed-notification boundary is covered by
    appending S08 if it is absent. Returns the augmented example list."""
    base = curate(pool, eval_items)
    have = {c["id"] for c in base}
    s08 = next(c for c in pool["candidates"] if c["id"] == "S08")
    return base + [s08] if "S08" not in have else base

_aug = curate_with_coverage(POOL, EVAL)
_doc, _pred, _ = run_variant(order_easy_to_hard(_aug), EVAL)
validate_output(_doc)
_f1 = score(GOLD_MAP, _pred)["MACRO"]["f1"]
_wrong = [r["id"] for r in _doc["records"] if r["label"] != GOLD_MAP[r["id"]]]
print(f"augmented macro F1 = {_f1:.3f} | wrong = {_wrong}")
check("stretch1: targeted example reaches macro F1 1.000", round(_f1, 3) == 1.000)

### Stretch 2 - Dynamic K by token budget

Examples cost context. Write `pick_by_budget` that greedily keeps examples (in the given order)
until a token budget would be exceeded, estimating one example's tokens as its character count
divided by four (floor division). Compare macro F1 at a tight budget versus a loose one to see the
cost versus quality trade-off. The exact formula lives in the code cell contract.

In [ ]:
def pick_by_budget(examples, budget_tokens):
    """Greedily keep examples in order while the running token estimate stays within budget.
    Token estimate for one example is max(1, len(text) // 4). Returns (kept_list, tokens_used)."""
    kept, used = [], 0
    for e in examples:
        cost = max(1, len(e["text"]) // 4)
        if used + cost <= budget_tokens:
            kept.append(e); used += cost
    return kept, used

_ordered = order_easy_to_hard(CURATED)
for _budget in (24, 999):
    _kept, _used = pick_by_budget(_ordered, _budget)
    _d, _p, _ = run_variant(_kept, EVAL)
    _f = score(GOLD_MAP, _p)["MACRO"]["f1"]
    print(f"budget={_budget:3}  K={len(_kept)}  tokens~{_used:2}  macro_F1={_f:.3f}  ids={[e['id'] for e in _kept]}")
check("stretch2: tight budget (24) keeps 2 examples", pick_by_budget(_ordered, 24)[0].__len__() == 2)

### Stretch 3 - Minimal-pair stress test

Counter-examples work because one token can move a label. Write `minimal_pair_probe` that runs a
list of short reviews (each a single-token variation on the same stem) through the zero-shot model
and returns id-to-label. Confirm the four variants below land on four different labels.

In [ ]:
def minimal_pair_probe(texts):
    """Run each text through the zero-shot model and return {index: predicted_label}."""
    items = [{"id": f"MP{i}", "text": t} for i, t in enumerate(texts)]
    doc = parse_output(simulated_llm([], items))
    return {int(r["id"][2:]): r["label"] for r in doc["records"]}

_variants = [
    "Thanks for the update",       # praise
    "Why the update",              # question
    "Please add the update",       # feature_request
    "The update keeps crashing",   # bug_report
]
_res = minimal_pair_probe(_variants)
for i, t in enumerate(_variants):
    print(f"  {t:28} -> {_res[i]}")
check("stretch3: four variants yield four distinct labels", len(set(_res.values())) == 4)

## Appendix - Swapping in a real provider (read only)

To move from the simulated model to a hosted one, keep every function above and replace only the
call site. The shape below is illustrative; confirm the model id, the request fields, and the
structured-output settings against current provider documentation before class, since those move.

```python
# import anthropic
# client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
# MODEL = "claude-..."             # confirm the current model id at delivery time
#
# def real_llm(examples, eval_items):
#     prompt = build_prompt(examples, eval_items)
#     resp = client.messages.create(
#         model=MODEL, max_tokens=1024,
#         messages=[{"role": "user", "content": prompt}],
#     )
#     return resp.content[0].text        # then parse_output(...) exactly as before
```

Your `parse_output` and `validate_output` already handle a real model's habit of wrapping JSON in
prose and fences, which is the whole point of owning those two functions.